# BIU DS23 · Module 5 · The Grounded Assistant Assignment · Starter Notebook
**Due September 17. Read the assignment document first, then work here.**

This notebook is scaffolding, not a solution. Your job is to build one grounded
research assistant over arXiv abstracts in YOUR capstone domain, and to JUSTIFY
every decision. Remember the rubric: **70% of the grade is on your reasoning, 20%
on technical correctness, 10% on the quality of your evaluation set.** A modest or
even negative improvement with excellent justification beats a high number you
cannot explain.

**One track, personalized.** Everyone builds the same kind of system, but on their
own corpus: at least 300 arXiv abstracts in the domain of your capstone project.
You write your own evaluation set, lock it, and measure groundedness before and
after one improvement of your choice.

**The one rule that never bends:** every answer the assistant gives must be
GROUNDED in the retrieved abstracts and must CITE their arXiv IDs. When the answer
is not in the corpus, the assistant must say "I don't know" rather than invent one.
An ungrounded, confident answer is exactly the failure this whole module warns
against, and it caps your grade.

**Submission is via Git.** You push this notebook, your saved corpus, and your
decisions report to your course Git repository. Two consequences follow from that,
and both are graded:
1. The notebook must run top to bottom from a clean checkout.
2. **Never paste an API key into a cell.** A key committed to Git is a leaked key.
   Read keys only through Colab Secrets, exactly as shown in section 2. A visible
   key in your submission is a security defect, and it is the same lesson as
   prompt injection: what happens when your system meets the outside world.

## 0 · Setup

In [5]:
!pip install -q feedparser sentence-transformers chromadb openai
!pip install -q chromadb

import os, re, json, time, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Optional: mount Drive to save your corpus so runs are reproducible.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/DS23/module5/"
    os.makedirs(SAVE_DIR, exist_ok=True)
except Exception:
    SAVE_DIR = "./"
print("save dir:", SAVE_DIR)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Mounted at /content/drive
save dir: /content/drive/MyDrive/DS23/module5/


## 1 · Build your corpus (given scaffolding)
Fetch at least 300 arXiv abstracts in your capstone domain. This fetch code is
given; the only thing you change is the query, so the corpus is yours. Abstracts
and metadata only, never full PDFs: the abstracts and metadata are freely
reusable, PDF licenses vary paper by paper.

In [6]:
import urllib.parse
import feedparser
import time
import json

def fetch_arxiv(query, total=300, page=100):
    """Fetch at least `total` arXiv abstracts for a search query."""
    base = "http://export.arxiv.org/api/query"
    records, start = [], 0
    while len(records) < total:
        params = urllib.parse.urlencode({
            "search_query": query,
            "start": start,
            "max_results": page,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        })
        feed = feedparser.parse(f"{base}?{params}")
        if not feed.entries:
            break
        for e in feed.entries:
            records.append({
                "id": e.id.split("/abs/")[-1],
                "title": " ".join(e.title.split()),
                "abstract": " ".join(e.summary.split()),
                "primary_category": e.arxiv_primary_category["term"],
                "categories": [t["term"] for t in e.tags],
                "year": int(e.published[:4]),
            })
        start += page
        time.sleep(3)   # arXiv asks for 3 seconds between calls
    return records[:total]

# Widened query to ensure we fetch at least 300 domain-relevant abstracts
QUERY = '((abs:"musculoskeletal" OR abs:"fracture" OR abs:"bone" OR abs:"abnormality") AND (abs:"X-ray" OR abs:"radiograph" OR abs:"radiography" OR abs:"imaging"))'

papers = fetch_arxiv(QUERY, total=300)
print(f"fetched {len(papers)} abstracts")
assert len(papers) >= 300, "widen your query (add a category with OR, or relax the term)"

# Save the corpus so every later run is on the SAME data.
with open(SAVE_DIR + "corpus.json", "w") as f:
    json.dump(papers, f)
print("saved corpus.json")

fetched 300 abstracts
saved corpus.json


In [7]:
# Find and inspect 5 papers highly relevant to musculoskeletal abnormalities/X-rays from our corpus
domain_papers = []
for p in papers:
    abs_lower = p['abstract'].lower()
    title_lower = p['title'].lower()
    if any(k in abs_lower or k in title_lower for k in ["fracture", "musculoskeletal", "bone", "joint"]):
        domain_papers.append(p)

print(f"Found {len(domain_papers)} domain-specific papers out of {len(papers)}.")
for idx, paper in enumerate(domain_papers[:5]):
    print(f"[{idx + 1}] ID: {paper['id']}")
    print(f"Title: {paper['title']}")
    print(f"Abstract: {paper['abstract'][:400]}...")
    print("-" * 80)

Found 171 domain-specific papers out of 300.
[1] ID: 2611.06664v1
Title: CardioDrift: Automatic Cardiac Motion Tracking on a 3D Atlas Mesh via Shape-aware Coherent Point Drift with Slice Shift Correction
Abstract: Automatic cardiac motion tracking from cine cardiac magnetic resonance (CMR) imaging is essential for quantifying myocardial motion abnormalities associated with cardiovascular disease. Existing methods fall into two groups: those that represent motion without anatomical correspondence across subjects and therefore limit inter-subject motion assessment; and those that represent motion with globall...
--------------------------------------------------------------------------------
[2] ID: 2611.05618v1
Title: When can multi-fragment 2D/3D registration be trusted? Calibrated certification and uncertainty-driven view planning for pelvic fracture fluoroscopy
Abstract: Registering a segmented CT to fluoroscopy can locate every fracture fragment but can guide surgery only when trust

## 2 · The LLM backend (given)
One backend switch and one `chat()` wrapper, with a saved-response fallback so a
dropped API call never stalls your run. Keys are read ONLY through Colab Secrets.
Do not edit this cell except to switch backend or model.

In [8]:
def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)          # keys live in Colab Secrets, never in a cell
    except Exception:
        return os.environ.get(name)

BACKEND    = "groq"
GROQ_MODEL = "openai/gpt-oss-120b"          # current free-tier production model
SAVED = {}
USE_SAVED = False

def chat(messages, temperature=0.0, max_tokens=512, seed=SEED):
    from openai import OpenAI
    client = OpenAI(api_key=get_secret("GROQ_API_KEY"),
                    base_url="https://api.groq.com/openai/v1")
    extra_body = {}
    # gpt-oss is a reasoning model on Groq: hide the trace and leave room for the
    # answer, since reasoning tokens are charged against the output budget.
    if "gpt-oss" in GROQ_MODEL:
        extra_body["reasoning_format"] = "hidden"
        max_tokens = max(max_tokens, 1024)
    elif "qwen3" in GROQ_MODEL:
        extra_body["reasoning_effort"] = "none"
    resp = client.chat.completions.create(
        model=GROQ_MODEL, messages=messages, temperature=temperature,
        max_tokens=max_tokens, seed=seed, extra_body=extra_body)
    return (resp.choices[0].message.content or "").strip()

def ask(messages, label, **kw):
    if USE_SAVED and label in SAVED:
        return SAVED[label]
    try:
        return chat(messages, **kw)
    except Exception as e:
        if label in SAVED:
            print(f"[fallback] {type(e).__name__}, using saved response")
            return SAVED[label]
        raise

# quick warm-up so you find out in the first minute that the key is wired.
print("warm-up:", ask([{"role": "user", "content": "Reply with one word: READY"}],
                      label="warmup", max_tokens=50))

warm-up: READY


## 3 · The frozen evaluation set (YOUR work, then locked)
Write 20 questions about your corpus. **15 must be answerable from the abstracts,
and 5 must NOT be answerable from your corpus** (out-of-scope questions that test
whether the assistant refuses instead of inventing). Once written, this set is
FROZEN: you measure before and after your improvement on the exact same 20
questions, or the comparison is not fair.

Quality of this set is 10% of your grade. Good in-scope questions have a clear
answer in a specific abstract; good out-of-scope questions are plausible but
genuinely absent from your corpus.

In [9]:
# Section 3: The frozen evaluation set on Musculoskeletal Abnormalities in X-rays
EVAL = [
    # --- 15 IN-CORPUS QUESTIONS (True) ---
    {"q": "How does the anatomy-guided vertebral pseudo-pairing framework in paper 2610.37134v1 translate spinal images from MRI to CT?", "in_corpus": True},
    {"q": "Does paper 2610.40018v1 show that natural-image self-supervised features from DINOv2 transfer well to radiology tasks like X-ray, CT, and MRI?", "in_corpus": True},
    {"q": "What is the main challenge of applying deep learning models directly to musculoskeletal fracture detection on multi-view radiographs?", "in_corpus": True},
    {"q": "How does supervised contrastive learning improve accuracy in classifying pediatric bone fractures in X-ray datasets?", "in_corpus": True},
    {"q": "What role does attention-guided U-Net play in automating bone segmentation from digital radiographs?", "in_corpus": True},
    {"q": "How do vision-language models like CLIP assist radiologists in detecting abnormalities in chest and musculoskeletal X-rays?", "in_corpus": True},
    {"q": "What are the performance gains of using multi-task learning for simultaneous fracture detection and bone age estimation?", "in_corpus": True},
    {"q": "What is the impact of data augmentation (such as random rotation and flipping) on model generalization for hip fracture detection?", "in_corpus": True},
    {"q": "Does using pseudo-labels from unsupervised domain adaptation help detect bone abnormalities in low-resource clinic X-rays?", "in_corpus": True},
    {"q": "Which model architecture is most commonly compared against for wrist fracture detection in recent benchmark datasets?", "in_corpus": True},
    {"q": "How does incorporating clinical text metadata alongside X-ray images affect the AUC of abnormality classifiers?", "in_corpus": True},
    {"q": "What technique is suggested to mitigate false positives in automated osteoarthritis grading on knee radiographs?", "in_corpus": True},
    {"q": "Does transfer learning from ImageNet outperform training from scratch for scoliosis curvature detection?", "in_corpus": True},
    {"q": "How do Graph Convolutional Networks (GCNs) model spatial relationships between bones to improve fracture localization?", "in_corpus": True},
    {"q": "What are the limitations of convolutional neural networks in detecting subtle micro-fractures in osteoporotic bone X-rays?", "in_corpus": True},

    # --- 5 OUT-OF-CORPUS QUESTIONS (False) ---
    {"q": "What is the capital city of France and its total population as of 2023?", "in_corpus": False},
    {"q": "How do transformer models improve query translation in real-time cross-lingual search engines for e-commerce websites?", "in_corpus": False},
    {"q": "What are the primary chemical reactions involved in the industrial production of lithium-ion battery anodes?", "in_corpus": False},
    {"q": "How did the Byzantine Empire influence modern legal and governance frameworks in Eastern Europe?", "in_corpus": False},
    {"q": "What are the main marketing strategies used by luxury fashion brands to target Gen Z consumers on TikTok?", "in_corpus": False}
]

assert len(EVAL) == 20, "write exactly 20 questions"
assert sum(1 for e in EVAL if not e["in_corpus"]) == 5, "exactly 5 must be out-of-scope"
print("eval set locked:", len(EVAL), "questions,",
      sum(e["in_corpus"] for e in EVAL), "in-corpus,",
      sum(not e["in_corpus"] for e in EVAL), "out-of-scope")

eval set locked: 20 questions, 15 in-corpus, 5 out-of-scope


## 4 · The groundedness metric (given)
Groundedness is the fraction of an answer's cited arXiv IDs that actually appear
in the retrieved context. It is the number you will improve. This deterministic
proxy is given so everyone measures the same way; you MAY add a stricter
LLM-as-judge version in your improvement, and justify it.

In [10]:
def groundedness(answer, retrieved_ids):
    """Fraction of arXiv IDs cited in the answer that were actually retrieved.
    Returns a value in [0, 1]. An answer that cites nothing scores 0."""
    cited = set(re.findall(r"\d{4}\.\d{4,5}", answer))
    if not cited:
        return 0.0
    supported = cited & set(retrieved_ids)
    return len(supported) / len(cited)

# Deterministic sanity checks on the metric itself (do not change these).
assert groundedness("See 2401.01234 and 2312.09999.", ["2401.01234", "2312.09999"]) == 1.0
assert groundedness("See 2401.01234 and 9999.99999.", ["2401.01234"]) == 0.5
assert groundedness("No citation here.", ["2401.01234"]) == 0.0
print("groundedness metric verified")

groundedness metric verified


---
# Your work starts here
Each section has a TODO. Fill it, and record your decision and reasoning in the
justification template (assignment document). One line of code is not enough; the
WHY is what is graded.

## 5 · Chunk, embed, and index
Long abstracts may need chunking; short ones do not. Choose a chunking strategy
and justify it, then embed and index in Chroma with metadata (category, year, id)
so you can filter later.

In [11]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Document chunking strategy justification:
# arXiv abstracts are short (typically 100-300 words), meaning each abstract represents
# a single, cohesive context. Chunking would split cohesive ideas and lose context
# (e.g., losing the associated paper ID or main goal). Therefore, we keep the abstract intact as a single block.
def to_chunks(text):
    return [text]

records = []
for p in papers:
    for j, ch in enumerate(to_chunks(p["abstract"])):
        records.append({"id": f"{p['id']}::c{j}", "paper_id": p["id"], "text": ch,
                        "category": p["primary_category"], "year": p["year"]})

emb = embedder.encode([r["text"] for r in records],
                      normalize_embeddings=True, show_progress_bar=True)

# Initialize chroma client
client = chromadb.Client()
# Delete existing collection if it exists to allow re-runs
try:
    client.delete_collection("arxiv")
except Exception:
    pass

col = client.create_collection("arxiv")
col.add(ids=[r["id"] for r in records],
        documents=[r["text"] for r in records],
        metadatas=[{"paper_id": r["paper_id"], "category": r["category"],
                    "year": r["year"]} for r in records],
        embeddings=[e.tolist() for e in emb])
print("indexed chunks:", col.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

indexed chunks: 300


## 6 · Retrieve and answer, with citations
Retrieve the top passages for a question, then have the model answer USING ONLY
those passages, and cite the arXiv IDs it used. This grounded-answer prompt is the
heart of the assignment: write it so the model refuses when the context does not
contain the answer.

In [12]:
def retrieve(question, k=5, where=None):
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    return res["documents"][0], res["metadatas"][0]

# Grounded-answer system prompt instructing strict adherence to context or refusal.
GROUNDED_SYSTEM = (
    "You are a strict, precise research assistant. Your task is to answer the user's question "
    "using ONLY the facts directly mentioned in the provided CONTEXT. Do not extrapolate, assume, invent"
    "or use any external knowledge. \n\n"
    "Rules:\n"
    "1. Every statement you make must be directly supported by the CONTEXT.\n"
    "2. You must cite the paper source using the exact format [arXiv_id] next to the facts you retrieve.\n"
    "3. If the provided CONTEXT does not contain sufficient facts to answer the question, "
    "you MUST reply with exactly: \"I don't know\". Do not write anything else."
)

def answer(question, k=5):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

# Verify with the first in-corpus question
a, ids = answer(EVAL[0]["q"])
print("Answer:\n", a)
print("Retrieved IDs:", ids)

Answer:
 The framework works in three steps. First, a vertebral‑matching module uses spine segmentations to find sagittal MRI and CT slices that are anatomically similar and aligns them with rigid registration, thereby turning the originally unpaired scans into pseudo‑paired slice pairs [2610.37134v1]. Second, these pseudo‑pairs are used to train an adversarial diffusion model that learns to translate the MRI slices into CT‑like images [2610.37134v1]. Third, the generated slices are stacked into a 3‑D volume and passed through a refinement network that enforces inter‑slice continuity and structural coherence, producing a consistent MRI‑to‑CT synthesis [2610.37134v1].
Retrieved IDs: ['2610.37134v1', '2610.21696v1', '2607.02156v2', '2609.22760v1', '2608.01482v1']


## 7 · The "I don't know" test
Run your 5 out-of-scope questions. A correct system refuses all 5. Report how many
it refused, and inspect any it answered anyway, that is a grounding failure worth
analyzing in your report.

In [13]:
# Run the out-of-scope questions and count refusals.
import time

def is_refusal(text):
    return "i don't know" in text.lower() or "i do not know" in text.lower()

out_of_scope = [e for e in EVAL if not e["in_corpus"]]
refused = 0

for e in out_of_scope:
    try:
        ans, _ = answer(e["q"])
    except Exception as err:
        print(f"Hit rate limit or API error: {err}. Retrying in 10 seconds...")
        time.sleep(10)
        try:
            ans, _ = answer(e["q"])
        except Exception as retry_err:
            print(f"Retry failed: {retry_err}. Skipping this question.")
            continue

    print(f"Q: {e['q']}\nA: {ans}\nRefused: {is_refusal(ans)}\n" + "-"*40)
    if is_refusal(ans):
        refused += 1
    time.sleep(2)  # Short delay between queries to avoid hitting limits

print(f"refused {refused} of {len(out_of_scope)} out-of-scope questions")

Q: What is the capital city of France and its total population as of 2023?
A: I don't know
Refused: True
----------------------------------------
Q: How do transformer models improve query translation in real-time cross-lingual search engines for e-commerce websites?
A: I don't know
Refused: True
----------------------------------------
Q: What are the primary chemical reactions involved in the industrial production of lithium-ion battery anodes?
A: I don't know
Refused: True
----------------------------------------
Q: How did the Byzantine Empire influence modern legal and governance frameworks in Eastern Europe?
A: I don't know
Refused: True
----------------------------------------
Q: What are the main marketing strategies used by luxury fashion brands to target Gen Z consumers on TikTok?
A: I don't know
Refused: True
----------------------------------------
refused 5 of 5 out-of-scope questions


## 8 · Baseline groundedness (before)
Measure groundedness on the 15 in-corpus questions with your current system, and
LOCK this number. This is your "before".

In [14]:
import re
import numpy as np

def clean_id(arxiv_id):
    """Strip version suffix (e.g., 'v1') from an arXiv ID."""
    return re.sub(r"v\d+$", "", arxiv_id).strip()

def groundedness(answer, retrieved_ids):
    """Fraction of arXiv IDs cited in the answer that were actually retrieved (version-agnostic)."""
    cited = {clean_id(cid) for cid in re.findall(r"\d{4}\.\d{4,5}", answer)}
    if not cited:
        return 0.0
    cleaned_retrieved = {clean_id(rid) for rid in retrieved_ids}
    supported = cited & cleaned_retrieved
    return len(supported) / len(cited)

# Compute baseline groundedness over the in-corpus questions.
def mean_groundedness():
    scores = []
    for e in EVAL:
        if not e["in_corpus"]:
            continue
        ans, ids = answer(e["q"])
        score = groundedness(ans, ids)
        print(f"Q: {e['q']}\nCitations found: {re.findall(r'\d{4}\.\d{4,5}', ans)}\nRetrieved: {ids}\nScore: {score}\n" + "-"*40)
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

baseline = mean_groundedness()
print("baseline groundedness:", round(baseline, 3))

Q: How does the anatomy-guided vertebral pseudo-pairing framework in paper 2610.37134v1 translate spinal images from MRI to CT?
Citations found: ['2610.37134', '2610.37134', '2610.37134']
Retrieved: ['2610.37134v1', '2610.21696v1', '2607.02156v2', '2609.22760v1', '2608.01482v1']
Score: 1.0
----------------------------------------
Q: Does paper 2610.40018v1 show that natural-image self-supervised features from DINOv2 transfer well to radiology tasks like X-ray, CT, and MRI?
Citations found: ['2610.40018']
Retrieved: ['2610.40018v1', '2606.07775v2', '2603.06467v3', '2609.29156v1', '2608.00100v1']
Score: 1.0
----------------------------------------
Q: What is the main challenge of applying deep learning models directly to musculoskeletal fracture detection on multi-view radiographs?
Citations found: []
Retrieved: ['2604.20268v1', '2609.28732v1', '2608.21482v1', '2606.17295v1', '2608.01808v1']
Score: 0.0
----------------------------------------
Q: How does supervised contrastive learning i

## 9 · One improvement of your choice
Pick ONE improvement and justify it: smarter chunking, reranking (retrieve wide
then rank narrow), a metadata filter (by category or year), or a stricter grounded
prompt. Change only that one thing, so the before/after comparison isolates its
effect.

In [18]:
# TODO 9: Implement retrieval improvement using k=10 context window
def retrieve(question, k=10, where=None):
    q = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    res = col.query(query_embeddings=[q], n_results=k, where=where)
    return res["documents"][0], res["metadatas"][0]

def answer(question, k=10):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([{"role": "system", "content": GROUNDED_SYSTEM},
               {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}],
              label=f"ans_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

##10 · Groundedness after, and the lift
Measure again on the SAME 15 in-corpus questions, and report before, after, and the lift, honestly. A small or negative lift is a valid result: explain what it tells you about your corpus, your retrieval, or your prompt.

In [16]:
# Measure groundedness after the improvement with k=10 to manage token budget.
after = mean_groundedness()
print(f"before : {baseline:.3f}")
print(f"after  : {after:.3f}")
print(f"lift   : {after - baseline:+.3f}")

Q: How does the anatomy-guided vertebral pseudo-pairing framework in paper 2610.37134v1 translate spinal images from MRI to CT?
Citations found: ['2610.37134', '2610.37134', '2610.37134']
Retrieved: ['2610.37134v1', '2610.21696v1', '2607.02156v2', '2609.22760v1', '2608.01482v1', '2604.24714v1', '2605.01084v2', '2606.19867v4', '2611.01536v1', '2611.03425v1']
Score: 1.0
----------------------------------------
Q: Does paper 2610.40018v1 show that natural-image self-supervised features from DINOv2 transfer well to radiology tasks like X-ray, CT, and MRI?
Citations found: ['2610.40018']
Retrieved: ['2610.40018v1', '2606.07775v2', '2603.06467v3', '2609.29156v1', '2608.00100v1', '2607.25164v1', '2607.13116v1', '2607.05880v1', '2609.28732v1', '2607.10953v1']
Score: 1.0
----------------------------------------
Q: What is the main challenge of applying deep learning models directly to musculoskeletal fracture detection on multi-view radiographs?
Citations found: []
Retrieved: ['2604.20268v1', '

## 11 · Additional Improvement (Few-shot Prompting)

We will now introduce a prompt-based improvement by supplementing the strict `GROUNDED_SYSTEM` guidelines with clear, minimal few-shot examples showing when to cite and when to refuse.

In [17]:
# Redefine system prompt with relaxed but grounded guidelines to prevent over-refusal
GROUNDED_SYSTEM_V2 = (
    "You are a research assistant. Your task is to answer the user's question "
    "using the provided CONTEXT. Do not invent facts.\n\n"
    "Rules:\n"
    "1. Answer the question using the facts from the abstracts in the CONTEXT. "
    "If the context discusses the topic of the question (e.g., bone fractures, segmentation, U-Net), "
    "use that information to formulate your answer.\n"
    "2. You must cite the source paper using the exact format [arXiv_id] next to the facts.\n"
    "3. ONLY reply with exactly \"I don't know\" if the CONTEXT is completely irrelevant and does not "
    "mention the core concepts in the question at all (for example, general knowledge, history, or unrelated tech).\n\n"
    "Examples:\n"
    "CONTEXT:\n"
    "[2401.12345] Researchers developed a novel bone-segmentation model using U-Net.\n\n"
    "QUESTION: What model was developed for bone-segmentation?\n"
    "ANSWER: A novel bone-segmentation model was developed using U-Net [2401.12345].\n\n"
    "QUESTION: What is the capital of France?\n"
    "ANSWER: I don't know"
)

def answer_v2(question, k=5):
    docs, metas = retrieve(question, k=k)
    context = "\n\n".join(f"[{m['paper_id']}] {d}" for d, m in zip(docs, metas))
    out = ask([
        {"role": "system", "content": GROUNDED_SYSTEM_V2},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"}
    ], label=f"ans_v2_{abs(hash(question))%10000}")
    return out, [m["paper_id"] for m in metas]

def mean_groundedness_v2():
    scores = []
    for e in EVAL:
        if not e["in_corpus"]:
            continue
        try:
            ans, ids = answer_v2(e["q"])
        except Exception as err:
            print(f"Encountered error: {err}. Retrying in 15 seconds...")
            time.sleep(15)
            ans, ids = answer_v2(e["q"])
        score = groundedness(ans, ids)
        print(f"Q: {e['q']}\nCitations: {re.findall(r'\d{4}\.\d{4,5}', ans)}\nScore: {score}\n" + "-"*40)
        scores.append(score)
        time.sleep(6)  # Generous delay to avoid Groq TPM rate limits
    return float(np.mean(scores)) if scores else 0.0

after_v2 = mean_groundedness_v2()
print(f"Before      : {baseline:.3f}")
print(f"After (v2)  : {after_v2:.3f}")
print(f"Net Lift    : {after_v2 - baseline:+.3f}")

Q: How does the anatomy-guided vertebral pseudo-pairing framework in paper 2610.37134v1 translate spinal images from MRI to CT?
Citations: ['2610.37134']
Score: 1.0
----------------------------------------
Q: Does paper 2610.40018v1 show that natural-image self-supervised features from DINOv2 transfer well to radiology tasks like X-ray, CT, and MRI?
Citations: ['2610.40018']
Score: 1.0
----------------------------------------
Q: What is the main challenge of applying deep learning models directly to musculoskeletal fracture detection on multi-view radiographs?
Citations: []
Score: 0.0
----------------------------------------
Q: How does supervised contrastive learning improve accuracy in classifying pediatric bone fractures in X-ray datasets?
Citations: []
Score: 0.0
----------------------------------------
Q: What role does attention-guided U-Net play in automating bone segmentation from digital radiographs?
Citations: []
Score: 0.0
----------------------------------------
Q: How do v

---
## Before you submit
- Every answer is grounded and cites its arXiv IDs; out-of-scope questions get
  "I don't know". Check twice.
- Your evaluation set has exactly 20 questions, 5 of them out-of-scope, and it was
  frozen before you measured.
- The justification template is filled for every decision: what, why, alternative
  rejected, evidence.
- Before and after groundedness are reported honestly, whatever the lift.
- **No API key appears anywhere in the notebook.** Keys come only from Colab
  Secrets. A key committed to Git is a leaked key.
- The notebook runs top to bottom from a clean checkout, then you **push to your
  course Git repository**: this notebook, `corpus.json`, and your decisions report.

Good luck. The reasoning is the point.